EGM722 Programming for GIS and Remote Sensing Assessment 2

Title: Mapping geothermal energy opportunity in relation to groundwater and environmental constraints in Northern Ireland using Python.

Aim: This project assesses the potential suitability of Northern Ireland aquifer geology for geothermal energy production by combining aquifer rock type information with underground temperature data. The analysis uses the principle that more porous aquifer materials are generally more favourable for geothermal energy systems because they can store and transmit groundwater more effectively. Areas with favourable aquifer geology and higher subsurface temperatures are identified as higher suitability zones.

1 Import Python libraries

In [5]:
import sys

!{sys.executable} -m pip install geopandas folium contextily

   ---------------------------------------- 0.0/342.5 kB ? eta -:--:--
   -------------------- ------------------- 174.1/342.5 kB 5.3 MB/s eta 0:00:01
   ---------------------------------------- 342.5/342.5 kB 5.3 MB/s eta 0:00:00
   ---------------------------------------- 0.0/113.4 kB ? eta -:--:--
   ---------------------------------------- 113.4/113.4 kB 6.9 MB/s eta 0:00:00
   ---------------------------------------- 0.0/22.9 MB ? eta -:--:--
   - -------------------------------------- 0.6/22.9 MB 12.9 MB/s eta 0:00:02
   -- ------------------------------------- 1.5/22.9 MB 16.1 MB/s eta 0:00:02
   ---- ----------------------------------- 2.4/22.9 MB 18.8 MB/s eta 0:00:02
   ----- ---------------------------------- 3.2/22.9 MB 18.8 MB/s eta 0:00:02
   ------- -------------------------------- 4.2/22.9 MB 17.8 MB/s eta 0:00:02
   -------- ------------------------------- 5.1/22.9 MB 18.1 MB/s eta 0:00:01
   ---------- ----------------------------- 6.0/22.9 MB 18.3 MB/s eta 0:00:01
  

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-expr 1.1.0 requires dask==2024.5.0, but you have dask 2026.1.2 which is incompatible.


In [6]:
# File and folder handling
from pathlib import Path
import os
import zipfile
import urllib.request

# Data analysis
import pandas as pd
import numpy as np

# Spatial data handling
import geopandas as gpd

# Mapping and plotting
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.lines as mlines

# Interactive web mapping
import folium

# Cartographic basemaps
import contextily as ctx

# Spatial geometry operations
from shapely.geometry import box

# Optional: suppress warnings for cleaner notebook output
import warnings
warnings.filterwarnings("ignore")

# Display all dataframe columns in notebook outputs
pd.set_option("display.max_columns", 100)

2 Set file paths

This section defines the folder structure used throughout the project.

Separate folders are used for:
- raw downloaded datasets,
- processed GIS outputs,
- and exported figures/results.

Using a consistent folder structure improves reproducibility and organisation of spatial analysis workflows. The folders are automatically created if they do not already exist.

The project uses `pathlib.Path` because it provides a platform-independent and reliable method for handling file paths in Python.

In [25]:
from pathlib import Path

# Project folders
PROJECT_DIR = Path.cwd().parent

DATA_RAW = PROJECT_DIR / "data_raw"
DATA_PROCESSED = PROJECT_DIR / "data_processed"
OUTPUTS = PROJECT_DIR / "outputs"

# Create folders if needed
for folder in [DATA_RAW, DATA_PROCESSED, OUTPUTS]:
    folder.mkdir(exist_ok=True)

3 Define functions

In [27]:
def download_file(url, output_path):
    """
    Download a file from a URL if it does not already exist.

    Parameters
    ----------
    url : str
        URL of the file to download.

    output_path : pathlib.Path
        Location where the file will be saved.

    Returns
    -------
    pathlib.Path
        Path to the downloaded file.
    """

    # Skip download if file already exists locally
    if output_path.exists():
        print(f"File already exists: {output_path.name}")
        return output_path

    print(f"Downloading: {url}")

    urllib.request.urlretrieve(url, output_path)

    print(f"Saved to: {output_path}")

    return output_path

def load_vector(path, target_crs=TARGET_CRS, layer=None):
    """
    Load a vector GIS dataset and reproject it to a target CRS.

    Parameters
    ----------
    path : str or pathlib.Path
        Path to the GIS file, for example SHP, GeoJSON or GeoPackage.
    target_crs : str
        Coordinate reference system to reproject the dataset into.
    layer : str, optional
        Layer name to read when using a GeoPackage with multiple layers.

    Returns
    -------
    geopandas.GeoDataFrame
        Loaded and reprojected spatial dataset.
    """
    # Read the spatial file using GeoPandas.
    gdf = gpd.read_file(path, layer=layer)

    # Stop the workflow if the dataset has no CRS.
    if gdf.crs is None:
        raise ValueError(f"{path} has no CRS defined.")

    # Reproject to the chosen CRS so all datasets align spatially.
    return gdf.to_crs(target_crs)


def clean_geodataframe(gdf):
    """
    Clean a GeoDataFrame by removing missing geometries and repairing invalid ones.

    Parameters
    ----------
    gdf : geopandas.GeoDataFrame
        Input spatial dataset.

    Returns
    -------
    geopandas.GeoDataFrame
        Cleaned spatial dataset.
    """
    # Work on a copy so the original data is not modified accidentally.
    gdf = gdf.copy()

    # Remove rows without valid geometry.
    gdf = gdf[gdf.geometry.notna()]
    gdf = gdf[~gdf.geometry.is_empty]

    # Repair invalid polygon geometries using a zero-width buffer.
    gdf["geometry"] = gdf.geometry.buffer(0)

    return gdf


def normalise_column(series):
    """
    Normalise numeric values to a 0-1 range.

    Parameters
    ----------
    series : pandas.Series
        Numeric column to normalise.

    Returns
    -------
    pandas.Series
        Values scaled between 0 and 1.
    """
    # Avoid division by zero if all values are the same.
    if series.max() == series.min():
        return series * 0

    return (series - series.min()) / (series.max() - series.min())


def classify_geothermal_suitability(description):
    """
    Classify aquifer geology into an indicative geothermal suitability score.

    The classification is based on the simplified assumption that more porous
    rock types are generally more favourable for geothermal production because
    they can store and transmit groundwater more effectively.

    Parameters
    ----------
    description : str
        Rock type, aquifer type or lithological description.

    Returns
    -------
    int
        Suitability score from 0 to 5, where 5 is highest suitability.
    """
    # Missing descriptions cannot be classified.
    if pd.isna(description):
        return 0

    # Convert text to lower case to make keyword matching easier.
    text = str(description).lower()

    # Sandstone is given the highest score because it is commonly porous.
    if "sandstone" in text:
        return 5

    # Limestone can be favourable, particularly where fractured or karstic.
    elif "limestone" in text or "chalk" in text:
        return 4

    # Volcanic rocks are variable, depending on fracturing and lithology.
    elif "basalt" in text or "rhyolite" in text or "volcanic" in text:
        return 3

    # Plutonic rocks generally have low primary porosity.
    elif "granite" in text or "diorite" in text or "gabbro" in text:
        return 2

    # Metamorphic rocks are generally lower suitability unless fractured.
    elif "schist" in text or "gneiss" in text or "quartzite" in text or "metamorphic" in text:
        return 1

    # Any unclassified geology is given a low-moderate default score.
    else:
        return 2


def classify_final_score(score):
    """
    Convert a combined numeric geothermal score into a suitability class.

    Parameters
    ----------
    score : float
        Combined geothermal suitability score.

    Returns
    -------
    str
        Suitability class: High, Moderate, Low or Very low.
    """
    if score >= 4:
        return "High"
    elif score >= 3:
        return "Moderate"
    elif score >= 2:
        return "Low"
    else:
        return "Very low"

4 Download and extract datasets from URLs

4.1 Download and extract aquifer data

In [33]:
PROJECT_DIR = Path.cwd().parent
DATA_RAW = PROJECT_DIR / "data_raw"
DATA_RAW.mkdir(exist_ok=True)

AQUIFER_URL = "https://gsni-data.bgs.ac.uk/geonetwork/srv/api/records/eeee2244-953d-4b81-8124-9e1ce602bf2e/attachments/NorthernIrelandsGroundwaterEnvironment.zip"

AQUIFER_ZIP = DATA_RAW / "aquifers.zip"
AQUIFER_EXTRACTED = DATA_RAW / "aquifers_extracted"

# Download the ZIP file
urllib.request.urlretrieve(AQUIFER_URL, AQUIFER_ZIP)

# Extract the ZIP file
AQUIFER_EXTRACTED.mkdir(exist_ok=True)

with zipfile.ZipFile(AQUIFER_ZIP, "r") as zip_ref:
    zip_ref.extractall(AQUIFER_EXTRACTED)

# List extracted files
for file in AQUIFER_EXTRACTED.rglob("*"):
    print(file)

C:\Users\User\EGM722_Assessment2_Diane_Gibson\data_raw\aquifers_extracted\NorthernIrelandsGroundwaterEnvironment
C:\Users\User\EGM722_Assessment2_Diane_Gibson\data_raw\aquifers_extracted\NorthernIrelandsGroundwaterEnvironment\ArcProLyrx
C:\Users\User\EGM722_Assessment2_Diane_Gibson\data_raw\aquifers_extracted\NorthernIrelandsGroundwaterEnvironment\AttributeDictionary.pdf
C:\Users\User\EGM722_Assessment2_Diane_Gibson\data_raw\aquifers_extracted\NorthernIrelandsGroundwaterEnvironment\LayerDefinitionFile
C:\Users\User\EGM722_Assessment2_Diane_Gibson\data_raw\aquifers_extracted\NorthernIrelandsGroundwaterEnvironment\NorthernIrelandsGroundwaterEnvironmentV1.2.gpkg
C:\Users\User\EGM722_Assessment2_Diane_Gibson\data_raw\aquifers_extracted\NorthernIrelandsGroundwaterEnvironment\QGISStyleFile
C:\Users\User\EGM722_Assessment2_Diane_Gibson\data_raw\aquifers_extracted\NorthernIrelandsGroundwaterEnvironment\ArcProLyrx\BedrockAquifer.lyrx
C:\Users\User\EGM722_Assessment2_Diane_Gibson\data_raw\aquife

4.2 Locate the Geopackage from the zip file

In [35]:
# Find GeoPackage files inside extracted folder
gpkg_files = list(AQUIFER_EXTRACTED.rglob("*.gpkg"))

print("GeoPackage files found:")
for file in gpkg_files:
    print(file)

AQUIFER_FILE = gpkg_files[0]
print("Using:", AQUIFER_FILE)

GeoPackage files found:
C:\Users\User\EGM722_Assessment2_Diane_Gibson\data_raw\aquifers_extracted\NorthernIrelandsGroundwaterEnvironment\NorthernIrelandsGroundwaterEnvironmentV1.2.gpkg
Using: C:\Users\User\EGM722_Assessment2_Diane_Gibson\data_raw\aquifers_extracted\NorthernIrelandsGroundwaterEnvironment\NorthernIrelandsGroundwaterEnvironmentV1.2.gpkg


4.3 Load the aquifer data

In [37]:
AQUIFER_URL = "https://gsni-data.bgs.ac.uk/geonetwork/srv/api/records/eeee2244-953d-4b81-8124-9e1ce602bf2e/attachments/NorthernIrelandsGroundwaterEnvironment.zip"
TEMPERATURE_URL = "https://webapps.bgs.ac.uk/services/ngdc/accessions/index.html#item185538"

# Output filenames
AQUIFER_FILE = DATA_RAW / "aquifers.gpkg"
TEMPERATURE_FILE = DATA_RAW / "temperature.gpkg"

# Download datasets
download_file(AQUIFER_URL, AQUIFER_FILE)
download_file(TEMPERATURE_URL, TEMPERATURE_FILE)

# Confirm downloads
print("\nAquifer exists:", AQUIFER_FILE.exists())
print("Temperature exists:", TEMPERATURE_FILE.exists())

File already exists: aquifers.gpkg
File already exists: temperature.gpkg

Aquifer exists: True
Temperature exists: True


5 Set Coordinate Reference System (CRS)

In [19]:
# Irish National Grid is appropriate for Northern Ireland spatial analysis.
TARGET_CRS = "EPSG:29903"

# WGS84 is required by Folium web maps.
WEB_CRS = "EPSG:4326"

6 Exploratory data analysis

6.1 Inspect the structure of the data

6.2 Check CRS

6.3 Summary statistics

7 Data cleaning

8 Feature selection

9 Check for missing data

10 Save the cleaned data

11 Reproject data to the same CRS

12 Clip datasets to Northern Ireland

13 Spatial analysis

14 Output map

15 Interactive map

16 Export outputs

17 Conclusions and limitations